# Deneyler 1–3: Eğitim aralığı, frame uzunluğu, robustness

Bildirideki sonuçlar bu notebook ile üretildi.

**Kullanım:** Çalışma zamanı → T4 GPU, sonra Tümünü çalıştır. Süre: ~35–45 dk.

Sonuçlar: `results/figures/exp*.png`, `results/tables/exp*.csv`

In [ ]:
# ==== HÜCRE 1: Kurulum ====
import os, sys

REPO_URL = "https://github.com/rftbrkzkn/snr-estimation-dl"
REPO_DIR = "/content/snr-estimation-dl" if os.path.exists("/content") else os.path.abspath("snr-estimation-dl")

%cd {os.path.dirname(REPO_DIR)}
!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

from src import config
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK (CPU) — T4 GPU seç")


In [ ]:
# ==== HÜCRE 2: Yardımcı fonksiyonlar ====
from src.data_gen import qpsk_symbols, add_awgn, generate_test_set, snr_test_grid, to_iq
from src.classical import m2m4
from src.features import extract_features, Standardizer
from src.models import FeatureMLP, CNN1D
from src.train import train_model, predict, set_seed
from src.evaluate import per_snr_metrics
from src.plots import (plot_metric_vs_snr, plot_pred_vs_true, plot_training_curves,
                       set_style, save, METHOD_COLORS, METHOD_MARKERS, INK)

TRAIN_SNR = (-15.0, 25.0)       # eğitim aralığı test aralığından (-10..20) geniş: kenar etkisi azalır
N_TR, N_VA, N_TE = config.N_TRAIN, config.N_VAL, config.N_TEST_PER_SNR
EP_F = config.EPOCHS
EP_C = config.EPOCHS if config.QUICK else 60
CNN_LR, CNN_PATIENCE = 3e-4, 12   # CNN: daha düşük learning rate, daha sabırlı early stopping
CFO_MAX = 0.005                  # frekans kayması: örnek başına en fazla ±0.005 devir

def impair(r, rng, phase=False, cfo=0.0):
    """Faz kayması (rastgele sabit açı) ve/veya frekans kayması (CFO) ekler."""
    if not phase and cfo == 0:
        return r
    n, N = r.shape
    th = rng.uniform(0, 2 * np.pi, n) if phase else np.zeros(n)
    f = rng.uniform(-cfo, cfo, n) if cfo > 0 else np.zeros(n)
    k = np.arange(N)
    return r * np.exp(1j * (th[:, None] + 2 * np.pi * f[:, None] * k[None, :]))

def make_train(n, N, rng, phase=False, cfo=0.0):
    snr = rng.uniform(*TRAIN_SNR, n)
    r = add_awgn(qpsk_symbols(n, N, rng), snr, rng)
    return impair(r, rng, phase, cfo), snr.astype(np.float32)

def make_test(N, rng, phase=False, cfo=0.0):
    rs, ys = [], []
    for s in snr_test_grid():
        r = add_awgn(qpsk_symbols(N_TE, N, rng), np.full(N_TE, s), rng)
        rs.append(impair(r, rng, phase, cfo))
        ys.append(np.full(N_TE, s))
    return np.concatenate(rs), np.concatenate(ys)

def fit_feature_mlp(r_tr, y_tr, r_va, y_va):
    F_tr = extract_features(r_tr)
    sc = Standardizer().fit(F_tr)
    m = FeatureMLP(F_tr.shape[1])
    h = train_model(m, sc.transform(F_tr), y_tr, sc.transform(extract_features(r_va)), y_va,
                    epochs=EP_F, verbose=False)
    print(f"  Feature + MLP: {len(h['val_loss'])} epoch, en iyi val RMSE {np.sqrt(min(h['val_loss'])):.3f} dB")
    return (m, sc), h

def fit_cnn(r_tr, y_tr, r_va, y_va):
    m = CNN1D()
    h = train_model(m, to_iq(r_tr), y_tr, to_iq(r_va), y_va,
                    epochs=EP_C, lr=CNN_LR, patience=CNN_PATIENCE, verbose=False)
    print(f"  1D CNN: {len(h['val_loss'])} epoch, en iyi val RMSE {np.sqrt(min(h['val_loss'])):.3f} dB")
    return m, h

def pred_fmlp(model, r):
    m, sc = model
    return predict(m, sc.transform(extract_features(r)))[0]

def pred_cnn(model, r):
    return predict(model, to_iq(r))[0]

def rmse_split(y, p):
    low = y <= -4
    f = lambda mask: float(np.sqrt(np.mean((p[mask] - y[mask]) ** 2)))
    return f(np.ones_like(low)), f(low), f(~low)

os.makedirs(config.TABLES_DIR, exist_ok=True)
print("Hazır. Eğitim SNR aralığı:", TRAIN_SNR, "| hızlı test modu:", config.QUICK)


---
## Deney 1 — AWGN, N = 1024
Eğitim −15…25 dB, test −10…20 dB. CNN: learning rate 3e-4, patience 12.

In [ ]:
# ==== HÜCRE 3: Deney 1 — kenar etkisi düzeltilmiş + CNN ayarlı (N = 1024, AWGN) ====
t_start = time.time()
set_seed()
rng = np.random.default_rng(1)
N = 1024

r_tr, y_tr = make_train(N_TR, N, rng)
r_va, y_va = make_train(N_VA, N, rng)
X_te, y_te = generate_test_set(N, N_TE)

print("Eğitim:")
fmlp_awgn, h_f = fit_feature_mlp(r_tr, y_tr, r_va, y_va)
cnn_awgn, h_c = fit_cnn(r_tr, y_tr, r_va, y_va)
del r_tr, r_va

preds1 = {"M2M4": m2m4(X_te),
          "Feature + MLP": pred_fmlp(fmlp_awgn, X_te),
          "1D CNN": pred_cnn(cnn_awgn, X_te)}
res1 = {k: per_snr_metrics(y_te, p) for k, p in preds1.items()}

plot_metric_vs_snr(res1, "rmse", name="exp1_rmse_vs_snr")
plot_metric_vs_snr(res1, "bias", name="exp1_bias_vs_snr")
plot_pred_vs_true(y_te, preds1, name="exp1_pred_vs_true")
plot_training_curves({"Feature + MLP": h_f, "1D CNN": h_c}, name="exp1_training_curves")
plt.show()

df1 = pd.DataFrame([dict(zip(["Yöntem", "RMSE tüm", "RMSE ≤ -4 dB", "RMSE > -4 dB"], [k, *rmse_split(y_te, p)]))
                    for k, p in preds1.items()])
df1.to_csv(f"{config.TABLES_DIR}/exp1_summary.csv", index=False)
pd.concat({k: d.set_index("snr_db") for k, d in res1.items()}, axis=1).to_csv(f"{config.TABLES_DIR}/exp1_per_snr.csv")
display(df1.round(3))
print(f"Deney 1 süresi: {(time.time() - t_start) / 60:.1f} dk")


---
## Deney 2 — Frame uzunluğu

In [ ]:
# ==== HÜCRE 4: Deney 2 — frame uzunluğu taraması (N = 128 / 256 / 512 / 1024) ====
t_start = time.time()
rows = []
for N in config.FRAME_LENS_SWEEP:
    print(f"N = {N}")
    X_n, y_n = generate_test_set(N, N_TE)
    if N == 1024:                     # Deney 1'de eğitildi, tekrar eğitmeye gerek yok
        p = preds1
    else:
        set_seed()
        rng = np.random.default_rng(100 + N)
        r_tr, y_tr = make_train(N_TR, N, rng)
        r_va, y_va = make_train(N_VA, N, rng)
        fm, _ = fit_feature_mlp(r_tr, y_tr, r_va, y_va)
        cn, _ = fit_cnn(r_tr, y_tr, r_va, y_va)
        del r_tr, r_va
        p = {"M2M4": m2m4(X_n), "Feature + MLP": pred_fmlp(fm, X_n), "1D CNN": pred_cnn(cn, X_n)}
    for k, pk in p.items():
        a, lo, hi = rmse_split(y_n, pk)
        rows.append({"N": N, "Yöntem": k, "RMSE tüm": a, "RMSE ≤ -4 dB": lo, "RMSE > -4 dB": hi})

df2 = pd.DataFrame(rows)
df2.to_csv(f"{config.TABLES_DIR}/exp2_frame_length.csv", index=False)

set_style()
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharex=True)
for ax, col in zip(axes, ["RMSE tüm", "RMSE ≤ -4 dB", "RMSE > -4 dB"]):
    for k in ["M2M4", "Feature + MLP", "1D CNN"]:
        d = df2[df2["Yöntem"] == k]
        ax.plot(d["N"], d[col], marker=METHOD_MARKERS[k], color=METHOD_COLORS[k], linewidth=2, markersize=5, label=k)
    ax.set_xscale("log", base=2)
    ax.set_xticks(config.FRAME_LENS_SWEEP)
    ax.set_xticklabels(config.FRAME_LENS_SWEEP)
    ax.set_yscale("log")
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("Frame uzunluğu N")
axes[0].set_ylabel("RMSE (dB)")
axes[0].legend()
fig.tight_layout()
save(fig, "exp2_rmse_vs_frame_length")
plt.show()

display(df2.pivot(index="N", columns="Yöntem", values="RMSE tüm").round(3))
print(f"Deney 2 süresi: {(time.time() - t_start) / 60:.1f} dk")


---
## Deney 3 — Faz ve frekans kayması

In [ ]:
# ==== HÜCRE 5: Deney 3 — robustness: faz ve frekans kayması (N = 1024) ====
t_start = time.time()
N = 1024
set_seed()
rng = np.random.default_rng(7)

# Bozucu etkili veriyle eğitilen modeller (augmentation)
print("Eğitim (faz + frekans kaymalı veri):")
r_tr, y_tr = make_train(N_TR, N, rng, phase=True, cfo=CFO_MAX)
r_va, y_va = make_train(N_VA, N, rng, phase=True, cfo=CFO_MAX)
fmlp_aug, _ = fit_feature_mlp(r_tr, y_tr, r_va, y_va)
cnn_aug, _ = fit_cnn(r_tr, y_tr, r_va, y_va)
del r_tr, r_va

scenarios = {"AWGN": {}, "Faz kayması": {"phase": True}, "Faz + frekans kayması": {"phase": True, "cfo": CFO_MAX}}
variants = {   # ad: (temel yöntem, model, tahmin fonksiyonu, AWGN ile mi eğitildi)
    "M2M4":                   ("M2M4", None, None, None),
    "Feature + MLP (AWGN)":   ("Feature + MLP", fmlp_awgn, pred_fmlp, True),
    "Feature + MLP (bozucu)": ("Feature + MLP", fmlp_aug, pred_fmlp, False),
    "1D CNN (AWGN)":          ("1D CNN", cnn_awgn, pred_cnn, True),
    "1D CNN (bozucu)":        ("1D CNN", cnn_aug, pred_cnn, False),
}

rows, curves = [], {}
for sname, kw in scenarios.items():
    X_s, y_s = make_test(N, np.random.default_rng(11), **kw)
    for vname, (base, model, fn, _) in variants.items():
        p = m2m4(X_s) if model is None else fn(model, X_s)
        a, lo, hi = rmse_split(y_s, p)
        rows.append({"Senaryo": sname, "Yöntem": vname, "RMSE tüm": a, "RMSE ≤ -4 dB": lo, "RMSE > -4 dB": hi})
        curves[(sname, vname)] = per_snr_metrics(y_s, p)

df3 = pd.DataFrame(rows)
df3.to_csv(f"{config.TABLES_DIR}/exp3_robustness.csv", index=False)

set_style()
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
for ax, sname in zip(axes, scenarios):
    for vname, (base, _, _, awgn_trained) in variants.items():
        d = curves[(sname, vname)]
        ax.plot(d["snr_db"], d["rmse"], color=METHOD_COLORS[base], marker=METHOD_MARKERS[base],
                markersize=4, linewidth=1.2 if awgn_trained else 2,
                linestyle="--" if awgn_trained else "-", label=vname)
    ax.set_yscale("log")
    ax.set_title(sname, fontsize=10)
    ax.set_xlabel("Gerçek SNR (dB)")
axes[0].set_ylabel("RMSE (dB)")
axes[-1].legend(fontsize=7, title="kesikli = AWGN ile eğitilmiş", title_fontsize=7)
fig.tight_layout()
save(fig, "exp3_robustness")
plt.show()

display(df3.pivot(index="Yöntem", columns="Senaryo", values="RMSE tüm")[list(scenarios)].round(3))
print(f"Deney 3 süresi: {(time.time() - t_start) / 60:.1f} dk")


---
## Sonuçları indir

In [ ]:
# ==== HÜCRE 6: Sonuçları indir ====
import shutil
zip_path = shutil.make_archive(os.path.join(os.path.dirname(REPO_DIR), "results_v2"), "zip", REPO_DIR, "results")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Colab dışında: dosya burada ->", zip_path)
